In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

STUDENT_ID = "TUPM-26-7958"
SEED_DIGIT = int(STUDENT_ID[-1])
RANDOM_SEED = int("".join(STUDENT_ID.split("-")[1:]))

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
if not DATA_DIR.is_dir():
    DATA_DIR = WORK_DIR.parent / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Load data
X = np.genfromtxt(DATA_DIR / "secom.data", dtype=float)
labels = np.genfromtxt(DATA_DIR / "secom_labels.data", dtype=str)
y = labels[:, 0].astype(float).astype(int)   # 1 = Fail, -1 = Pass

df = pd.DataFrame(X)
fail = y == 1
pas = y == -1

# 1. Select 5 usable features (>= 10 observed values in each group)
usable = [c for c in df.columns
          if df.loc[pas, c].notna().sum() >= 10
          and df.loc[fail, c].notna().sum() >= 10
          and df[c].std() > 0]
rng = np.random.default_rng(RANDOM_SEED)
features = sorted(rng.choice(usable, size=5, replace=False).tolist())
data = df[features]

# 2-3. Two strategies: available-case and median imputation
avail = data.copy()
imputed = data.fillna(data.median())

avail_diff = avail[fail].mean() - avail[pas].mean()
imp_diff = imputed[fail].mean() - imputed[pas].mean()

# 4-5. Stratified bootstrap
n_boot = 200 + 10 * SEED_DIGIT

def bootstrap(d):
    rng = np.random.default_rng(RANDOM_SEED)
    f_arr = d[fail].to_numpy()
    p_arr = d[pas].to_numpy()
    diffs = []
    for _ in range(n_boot):
        f = f_arr[rng.integers(0, len(f_arr), len(f_arr))]
        p = p_arr[rng.integers(0, len(p_arr), len(p_arr))]
        diffs.append(np.nanmean(f, axis=0) - np.nanmean(p, axis=0))
    diffs = np.array(diffs)
    return diffs.mean(axis=0), np.percentile(diffs, 2.5, axis=0), np.percentile(diffs, 97.5, axis=0)

a_mean, a_low, a_high = bootstrap(avail)
i_mean, i_low, i_high = bootstrap(imputed)

# 6. Compare strategies
result = pd.DataFrame({
    "Feature": features,
    "Avail_Diff": avail_diff.values,
    "Avail_Boot_Mean": a_mean,
    "Avail_CI_Low": a_low,
    "Avail_CI_High": a_high,
    "Imputed_Diff": imp_diff.values,
    "Imputed_Boot_Mean": i_mean,
    "Imputed_CI_Low": i_low,
    "Imputed_CI_High": i_high,
})
result["Change"] = (result["Imputed_Diff"] - result["Avail_Diff"]).abs()
result["Same_Direction"] = np.sign(result["Avail_Diff"]) == np.sign(result["Imputed_Diff"])
result["Avail_Rank"] = result["Avail_Diff"].abs().rank(ascending=False)
result["Imputed_Rank"] = result["Imputed_Diff"].abs().rank(ascending=False)

# 7. Reproducibility check and export
check = bootstrap(avail)
reproducible = np.allclose(a_mean, check[0])

result.to_csv(OUTPUT_DIR / "robustness_comparison.csv", index=False)

top = result.loc[result["Change"].idxmax()]
report = f"""SENSITIVITY REPORT
Student ID            : {STUDENT_ID}
Selected Features     : {features}
Bootstrap Repetitions : {n_boot}
Largest Change        : Feature {top['Feature']} ({top['Change']:.6f})
All Same Direction    : {result['Same_Direction'].all()}
Rankings Identical    : {(result['Avail_Rank'] == result['Imputed_Rank']).all()}
Reproducible          : {reproducible}
"""
(OUTPUT_DIR / "sensitivity_report.txt").write_text(report)

print(report)
print(result.round(4).to_string(index=False))

SENSITIVITY REPORT
Student ID            : TUPM-26-7958
Selected Features     : [34, 117, 201, 331, 545]
Bootstrap Repetitions : 280
Largest Change        : Feature 201 (0.004976)
All Same Direction    : True
Rankings Identical    : True
Reproducible          : True

 Feature  Avail_Diff  Avail_Boot_Mean  Avail_CI_Low  Avail_CI_High  Imputed_Diff  Imputed_Boot_Mean  Imputed_CI_Low  Imputed_CI_High  Change  Same_Direction  Avail_Rank  Imputed_Rank
      34      0.0120           0.0064       -0.1823         0.2793        0.0122             0.0066         -0.1823           0.2793  0.0001            True         4.0           4.0
     117     -0.4355          -0.4105       -0.8660        -0.0644       -0.4355            -0.4105         -0.8660          -0.0644  0.0000            True         2.0           2.0
     201      0.5894           0.6127       -0.3469         1.9290        0.5943             0.6178         -0.3414           1.9322  0.0050            True         1.0           1.0
